**你们为什么选 Trino？它跟 Spark SQL、Hive 这些引擎比起来，定位有什么不同？**

先说 Hive： Hive 是基于 MapReduce 的，延迟太高了，一个简单的 count 都要几十秒，而且只支持批处理，不适合交互式查询。我们现在基本只用 Hive 做元数据管理，查询已经不用了。
再说 Spark SQL： Spark SQL 其实也很强大，我们用它做批处理 ETL，跑 DWD、DWS 层的数仓任务。但它有个特点是批处理优先，每次查询要启动 Spark 应用，有初始化开销，而且资源调度比较重。
我再说说我自己做的这个项目，我要求时效性能做到每10分钟计算一次，第12分钟数据可见，这种是典型的高频upsert场景，trino的执行模式会更适合，首先，Spark启动开销太大太久，已经不符合时延要求了，我之前通过kyuubi复用了一个spark引擎解决了启动开销的问题，但是呢Spark不是为该场景设计的，你每个提交的SQL都会走重量级的优化器解析规划阶段，你多提交几个SQL就可能把Spark Driver干崩了，还没有细粒度的资源组控制，你不得不去加一些排队规划的逻辑，但是你即使加上了排队，但仍然可能遇到几个大的SQL把executor打崩，处理起来特别麻烦。
然后再说一下Trino，MPP架构的，纯内存计算，数据在内存中流转，不需要落地，所以延迟特别低，性能也别快，而且它是常驻引擎，因此先天就屏蔽了启动延迟问题，然后是连接器生态特别丰富，大部分数据它都能读，再者它对Iceberg的支持几乎可以覆盖我们目前所有的需求，即使不能覆盖，也可以通过Spark SQL补充。

**Trino 执行一条 SQL 的完整流程是怎样的？**

第一阶段，SQL解析，Coordinator 收到 SQL 后，先用 ANTLR 解析器把 SQL 字符串解析成 AST，这个阶段如果 SQL 语法有错误，会直接报错，比如少了个逗号、关键字拼错了。
第二阶段：语义分析，这一步会做几件事：元数据校验，类型推导，权限校验，如果表不存在或者权限不足，这个阶段就会报错。
第三阶段：逻辑计划生成与优化，AST 会被转换成逻辑执行计划，然后经过一系列优化规则，规则大概有谓词下推，列裁剪，常量折叠，Limit 下推等优化
第四阶段：物理计划阶段：将逻辑计划转换为具体的执行计划，包括任务分配、数据分区策略等执行细节。
最后就是任务执行了。

**能详细说说 Trino 的 Connector 架构吗？Iceberg Connector 是怎么工作的？**

Connector 是 Trino 最核心的扩展机制，Trino 本身不存储数据，所有数据读写都通过 Connector，Connector 架构分三层，第一层是Trino 定义的一套标准接口，包括ConnectorMetadata，用于提供表的元数据（Schema、列、分区等），ConnectorSplitManager，负责数据分片（Split），决定怎么并行读取，ConnectorPageSourceProvider，提供数据读取器，返回 Page，ConnectorPageSinkProvider，提供数据写入器（INSERT 时用），所有 Connector 都要实现这些接口，这样 Trino 才能统一管理，第二层是具体 Connector 实现，比如 Iceberg Connector、MySQL Connector、Hive Connector，每个都实现上面的 SPI。比如 Iceberg 表、MySQL 数据库、HDFS 文件等。我以 Iceberg Connector 为例，说一下查询流程：
第一步：元数据加载
当你查询 SELECT * FROM iceberg.catalog.table 时，Trino 会通过 Iceberg Connector 的 Metadata 模块去读取 Iceberg 的元数据。
Iceberg 的元数据是分层的：
Metadata JSON：记录表的 Schema、分区、快照（Snapshot）信息
Manifest List：记录每次快照的 Manifest 文件列表
Manifest File：记录每个数据文件的元数据（分区、行数、列的 min/max 等）
Data File：实际的 Parquet/ORC 数据文件
Connector 会先读 Metadata JSON，找到当前快照，然后读 Manifest List，再读 Manifest File，最后得到数据文件列表。
第二步：分区裁剪
如果 SQL 有 WHERE 条件，比如 WHERE biz_date = '2024-01-01'，Connector 会利用 Manifest 里的分区信息，过滤掉不满足条件的分区，只读取相关的数据文件。
第三步：文件过滤
即使分区确定了，可能还有很多文件。Connector 会利用 Manifest 里的列统计信息（min/max、null count）过滤掉不包含目标数据的文件。
比如 WHERE user_id = 12345，如果某个文件的 user_id 范围是1至100，那这个文件就可以跳过。
第四步：读取数据
确定要读哪些文件后，SplitManager 会为每个文件（或者文件的一部分）创建一个 Split，分发给 Worker 并行读取。
Worker 通过 PageSourceProvider 创建 PageSource，读取 Parquet 文件，转换成 Trino 的 Page 格式返回。
Parquet 读取还有两个优化：
列裁剪：只读查询需要的列，Parquet 是列式存储，这个效果很明显
谓词下推：Parquet 的页级别统计信息可以过滤掉不满足条件的数据页
第五步：INSERT 写入流程（如果有的话）
当我们用 Trino 执行 INSERT INTO iceberg.table ... 时：
Connector 创建 PageSink
Worker 将数据写入临时文件（Parquet 格式）
写入完成后，更新 Manifest 和 Metadata JSON，创建新快照
注意：这里会触发 Iceberg 的小文件问题，所以我们定期跑 Compaction 合并小文件

**Trino 是内存计算，那它的内存是怎么管理的？跟 Spark 比有什么不同？**

Trino的内存模型从宏观上来看分两类，第一类是堆内存，第二类是堆内存预留空间，用于GC和系统开销，再说说堆内存，堆内存又分为用户内存，系统内存，以及堆预留空间，和Spark不一样的是Spark中将堆预留空间又分为存储内存和执行内存，并可以动态分配，Trino没有这么细，但是Trino可以严格限制每个查询可以使用的内存，超出限制内存会直接kill掉查询，Spark的话会溢写到磁盘或者直接报OOM，因为Trino少了动态分配因此也会对比Spark的内存分配速度要快一点。

**Trino 查询慢，你一般怎么排查和优化？**

第一步：我会先跑 EXPLAIN ANALYZE，看执行计划和实际统计信息，重点看几个指标：Input vs Output 比例：如果 TableScan 读了 10 亿行，最后只输出 1 万行，说明过滤不够早，耗时最长的 Operator：可能是 Join、Aggregate 或者数据倾斜，数据分布：看每个 Task 处理的数据量是否均匀
第二步：针对性优化，根据瓶颈不同，优化策略也不同：如果是TableScan 慢（读数据慢），可能原因：没做分区裁剪、没做谓词下推、文件太小（小文件问题）、文件格式不好（比如 CSV vs Parquet），分区裁剪和谓词下推就要检查where条件写得对不对，小文件问题的话就要检查Iceberg表的维护策略，然后是Join慢的话，就得看是不是大表 Join 大表，Shuffle 数据量大，或者数据倾斜，数据倾斜的话就得尝试过滤或者加盐，场景三的就是Exchange 慢，exchange是发生在网络中的，看看是不是Stage 之间传输数据量太大，太大的话就可以尝试一下减少中间结果集。

**生产环境肯定要考虑安全，Trino 的权限控制你们是怎么做的？**

安全这块我确实花了不少精力，但说实话我在这块做的并不深入，因为安全认证一般分为认证和鉴权以及传输安全，第一个步骤是要开启https加密，说实话，这个步骤我也了解得不多，我当初部署的时候几乎依赖于AI才勉强配置成功了，是通过k8s的cert-manager发放的凭证，等cert-manager创建了证书secrets以后，挂载到Trino中并在配置中启动后就可以了，然后是认证，我在k8s中部署了一个LLDAP服务，Trino和kyuubi公用这个lldap服务，但实际上一些ldap的高级特性我没有用，我目前的用法是通过lldap保存用户名和密码，再然后是鉴权，说实话我Trino的鉴权根本没做，但是Iceberg底层数据我是通过Rest Catalog做了用户只能访问哪些目录，库，和表的。